In [9]:
import nltk
import pickle
import string
import os
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tag import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.probability import FreqDist

# nltk.download('all') -> kalo gada aja

MODEL_PATH = "./models.pkl"
CSV_PATH = "./financial_dataset.csv"

In [10]:
df = pd.read_csv(CSV_PATH)
df.head()

,Statement,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,$SPY wouldn't be surprised to see a green close,positive
4,Shell's $70 Billion BG Deal Meets Shareholder ...,negative


In [11]:
df['Sentiment'].value_counts()

Sentiment
positive    1852
negative     860
Name: count, dtype: int64

In [12]:
stemmer = PorterStemmer()
wnl = WordNetLemmatizer()
eng_stopwords = stopwords.words("english")

In [14]:
def preprocess(doc):
    words = word_tokenize(doc.lower())

    words = [wnl.lemmatize(word) for word in words]
    words = [stemmer.stem(word) for word in words]

    return {word: True for word in words if word not in eng_stopwords and word.isalpha()}

In [15]:
# train model
def train_model():
    df = pd.read_csv("./financial_dataset.csv")

    feature_sets = [(preprocess(text), label) for text, label in zip(df["Statement"], df["Sentiment"])]

    shuffle(feature_sets)

    split_index = int(len(feature_sets) * .8)
    train_df, test_df = feature_sets[:split_index], feature_sets[split_index:]

    naive_bayes = nltk.NaiveBayesClassifier.train(train_df)

    accuracy = nltk.classify.accuracy(classifier=naive_bayes, gold=test_df)
    print("Accuracy: ", accuracy)

    naive_bayes.show_most_informative_features(5)

    model_file = open("./model.pickle", "wb")
    pickle.dump(naive_bayes, model_file)
    model_file.close()

    return naive_bayes

In [16]:
def read_model():
    try:
        model_file = open("./model.pickle", "rb")
        print("Model is available!")

        print("Loading the model...")
        model = pickle.load(model_file)
        model_file.close()

        print("Model load successfully!")
        model.show_most_informative_features(5)
    except:
        print("Model is not available!")
        print("Preparing for model training...")
        model = train_model()

    return model

In [17]:
def analyze_review(review, model):
    if len(review) == 0:
        print("Review is empty!")
        return

    words = word_tokenize(review.lower())

    words = FreqDist([word for word in words if word.isalpha() and word not in string.punctuation])

    tagged = pos_tag(words)

    print("POS Tag:")

    for word, tag in tagged:
        print(f"- {word}: {tag}")

    for word in words:
        print(f"Word: {word}")
        
        synsets = wordnet.synsets(word)
        synonyms = []
        antonyms = []

        for synset in synsets:
            for lemma in synset.lemmas():
                synonyms.append(lemma.name())
                for antonym in lemma.antonyms():
                    antonyms.append(antonym.name())
        
        print("Synonyms:")
        if len(synonyms) == 0:
            print("No synonym detected!")
        else:
            print(synonyms[:5])

        print("Antonyms:")
        if len(antonyms) == 0:
            print("No antonym detected!")
        else:
            print(antonyms[:5])

        print()
    
    clean_review = [word for word in word_tokenize(review) if word not in string.punctuation and word not in eng_stopwords]

    clean_review = [wnl.lemmatize(stemmer.stem(word)) for word in clean_review]

    result = model.classify(FreqDist(clean_review))

    print(f"Prediction sentiment: {result}")

In [18]:
def write_review():
    while True:
        review = input("Input your review [>= 2 words]: ")
        
        words = review.split()
        
        if len(words) > 1:
            print("Review added!")
            return review
        else:
            print("Your review must consists of at least 2 words!")

In [19]:
if __name__ == "__main__":
    model = None
    review = ""

    while True:
        print("1. Write your statement")
        print("2. Analyse your statement")
        print("3. Exit")

        choice = int(input("Enter your choice: "))

        if choice == 1:
            model = read_model()
            review = write_review()
        elif choice == 2:
            analyze_review(review, model)
        elif choice == 3:
            print("Thanks for using this application!")
            break
        else:
            print("Please enter a valid choice")

1. Write your statement
2. Analyse your statement
3. Exit
Model is available!
Loading the model...
Model load successfully!
Most Informative Features
                decrease = True           negati : positi =     31.8 : 1.0
                    fell = True           negati : positi =     30.4 : 1.0
                     lay = True           negati : positi =     19.8 : 1.0
                   staff = True           negati : positi =     18.4 : 1.0
                  recall = True           negati : positi =     16.3 : 1.0
Review added!
1. Write your statement
2. Analyse your statement
3. Exit
POS Tag:
- hello: NN
- there: EX
Word: hello
Synonyms:
['hello', 'hullo', 'hi', 'howdy', 'how-do-you-do']
Antonyms:
No antonym detected!

Word: there
Synonyms:
['there', 'there', 'at_that_place', 'in_that_location', 'there']
Antonyms:
['here', 'here', 'here']

Prediction sentiment: positive
1. Write your statement
2. Analyse your statement
3. Exit
Thanks for using this application!
